# 의사결정트리 (Decision Tree) — 16조 발표 노트북

> **빅데이터프로그래밍1 · 팀 프로젝트**
> 16조 · 팀원 4명
> _수업에서 배운 순수 파이썬 문법만으로 구현한 의사결정트리_

---

**한 줄 요약**: 데이터가 스스로 만들어내는 `if/else` 규칙 — 우리는 그 규칙을 만드는 알고리즘을 직접 짰습니다.

▶ **런타임 > 모두 실행** 하면 끝까지 자동으로 실행됩니다.

---

## 목차

1. 데이터 불러오기 (UCI Mushroom)
2. 직관: 사람의 `if`문 vs 데이터가 만든 트리
3. Gini 불순도 — 얼마나 섞여 있나?
4. `split` / `best_split` — 가장 좋은 분기 찾기
5. `Node` + `build_tree` — 재귀로 트리 자라기
6. `predict` + `print_tree` — 트리 시각화
7. `max_depth` 실험 — 깊이별 정확도
8. 한계와 정리


## 1. 데이터 불러오기

UCI **Mushroom** 데이터셋 (Kaggle 동일 원본)을 자동 다운로드합니다.

- 샘플 수: 8124
- 특징 수: 22개 (모두 범주형)
- 레이블: `e` (식용) / `p` (독성)
- 결측치 없음 (`?`는 stalk-root 일부에만 존재, 별도 카테고리로 취급)


In [1]:
import urllib.request, os

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
DATA_PATH = "mushroom.data"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("다운로드 완료:", DATA_PATH)
else:
    print("이미 존재:", DATA_PATH)

print("파일 크기(bytes):", os.path.getsize(DATA_PATH))


이미 존재: mushroom.data
파일 크기(bytes): 373704


In [2]:
FEATURE_NAMES = [
    "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring", "veil-type", "veil-color", "ring-number",
    "ring-type", "spore-print-color", "population", "habitat"
]


def load_data(path):
    data = []
    f = open(path, "r")
    for line in f:
        line = line.strip()
        if line == "":
            continue
        parts = line.split(",")
        label = parts[0]
        features = {}
        i = 0
        while i < len(FEATURE_NAMES):
            features[FEATURE_NAMES[i]] = parts[i + 1]
            i = i + 1
        sample = {"label": label, "features": features}
        data.append(sample)
    f.close()
    return data


data = load_data(DATA_PATH)
print("전체 샘플 수:", len(data))
print("특징 개수:", len(FEATURE_NAMES))

edible = sum(1 for s in data if s["label"] == "e")
poison = sum(1 for s in data if s["label"] == "p")
print(f"식용(e): {edible}개 ({edible/len(data)*100:.1f}%)")
print(f"독성(p): {poison}개 ({poison/len(data)*100:.1f}%)")
print("\n첫 번째 샘플 예시:")
print("  label =", data[0]["label"])
for k in list(data[0]["features"].keys())[:5]:
    print(f"  {k:20s} = {data[0]['features'][k]}")
print("  ... (총 22개 특징)")


전체 샘플 수: 8124
특징 개수: 22
식용(e): 4208개 (51.8%)
독성(p): 3916개 (48.2%)

첫 번째 샘플 예시:
  label = p
  cap-shape            = x
  cap-surface          = s
  cap-color            = n
  bruises              = t
  odor                 = p
  ... (총 22개 특징)


## 2. 직관: 사람의 `if`문 vs 데이터가 만든 트리

사람이 손으로 짠다면 이런 식이겠죠.

```python
if odor == "n":          # 냄새가 없으면
    if spore_color == "r":   # 포자색이 빨강이면
        return "독성"
    else:
        return "식용"
else:
    ...
```

**문제**: "어떤 특징을, 어떤 값에서, 어떤 순서로 물어볼지" — 사람이 일일이 정해야 함.

**의사결정트리의 아이디어**: 데이터가 스스로 답하게 한다.
> *"가장 깔끔하게 두 그룹으로 갈라주는 질문은 뭐야?"* 를 반복적으로 묻기.

이 "깔끔함"을 수치로 만든 게 **Gini 불순도**.


## 3. Gini 불순도 — 얼마나 섞여 있나?

$$ \text{Gini}(S) = 1 - \sum_i p_i^2 $$

- 한 그룹이 한 종류로만 채워져 있으면 → **0** (완벽)
- 두 종류가 50:50으로 섞여 있으면 → **0.5** (최악)

직관 예시: 사과/오렌지가 섞인 바구니의 불순도를 계산해 봅시다.


In [3]:
def gini(data):
    counts = {}
    for sample in data:
        lab = sample["label"]
        counts[lab] = counts.get(lab, 0) + 1
    total = len(data)
    impurity = 1.0
    for lab in counts:
        p = counts[lab] / total
        impurity = impurity - p * p
    return impurity


def make_fake(apple, orange):
    res = []
    for _ in range(apple):
        res.append({"label": "apple", "features": {}})
    for _ in range(orange):
        res.append({"label": "orange", "features": {}})
    return res


print("사과 10, 오렌지 0 (완벽)     -> gini =", round(gini(make_fake(10, 0)), 4))
print("사과 9,  오렌지 1            -> gini =", round(gini(make_fake(9, 1)), 4))
print("사과 7,  오렌지 3            -> gini =", round(gini(make_fake(7, 3)), 4))
print("사과 5,  오렌지 5 (최악)     -> gini =", round(gini(make_fake(5, 5)), 4))

print("\n전체 train 데이터의 gini:", round(gini(data), 4))


사과 10, 오렌지 0 (완벽)     -> gini = 0.0
사과 9,  오렌지 1            -> gini = 0.18
사과 7,  오렌지 3            -> gini = 0.42
사과 5,  오렌지 5 (최악)     -> gini = 0.5

전체 train 데이터의 gini: 0.4994


## 4. `split` / `best_split` — 가장 좋은 분기 찾기

**아이디어**: 특정 특징의 특정 값으로 데이터를 두 그룹으로 나눴을 때,
나뉜 두 그룹의 Gini를 가중평균 한 값이 **원래보다 얼마나 줄었는지**(= 정보 이득, gain)를 본다.

모든 (특징, 값) 조합을 시도해서 가장 큰 gain을 주는 분기를 고른다.


In [4]:
def split(data, feature, value):
    left = [s for s in data if s["features"][feature] == value]
    right = [s for s in data if s["features"][feature] != value]
    return left, right


def best_split(data):
    base_gini = gini(data)
    best_gain = 0.0
    best_feature = None
    best_value = None
    total = len(data)
    for feature in FEATURE_NAMES:
        values = set()
        for s in data:
            values.add(s["features"][feature])
        for value in values:
            left, right = split(data, feature, value)
            if len(left) == 0 or len(right) == 0:
                continue
            w_left = len(left) / total
            w_right = len(right) / total
            weighted = w_left * gini(left) + w_right * gini(right)
            gain = base_gini - weighted
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_value = value
    return best_feature, best_value, best_gain


### Train / Test 분리 (라이브러리 없이 재현 가능하게)

`i % 10 < 7` 이면 train, 아니면 test → 7:3 분할 + 항상 같은 결과.


In [5]:
train = []
test = []
i = 0
while i < len(data):
    if i % 10 < 7:
        train.append(data[i])
    else:
        test.append(data[i])
    i = i + 1

print("train:", len(train), "/ test:", len(test))
print("train의 base gini:", round(gini(train), 4))

f, v, g = best_split(train)
print(f"\n>>> 첫 분기: [{f} == {v}?]  (gain = {round(g, 4)})")
print("    → '냄새(odor)가 없는가?' 이 한 질문이 식용/독성을 가장 잘 나눕니다.")


train: 5688 / test: 2436
train의 base gini: 0.4991

>>> 첫 분기: [odor == n?]  (gain = 0.3095)
    → '냄새(odor)가 없는가?' 이 한 질문이 식용/독성을 가장 잘 나눕니다.


## 5. `Node` 클래스 + `build_tree` (재귀)

각 노드는 다음 중 하나입니다.
- **분기 노드**: `(feature, value)` 질문을 들고, `left`(YES)와 `right`(NO) 자식 노드를 가진다.
- **잎 노드**: `prediction` 라벨을 가진다.

종료 조건:
1. `gini == 0` (이미 한 종류만 있음)
2. `depth >= max_depth` (깊이 제한 도달)
3. 더 이상 gain > 0 인 분기를 못 찾음


In [6]:
class Node:
    def __init__(self):
        self.feature = None
        self.value = None
        self.left = None
        self.right = None
        self.prediction = None


def majority_label(data):
    counts = {}
    for s in data:
        lab = s["label"]
        counts[lab] = counts.get(lab, 0) + 1
    best_lab = None
    best_count = -1
    for lab in counts:
        if counts[lab] > best_count:
            best_count = counts[lab]
            best_lab = lab
    return best_lab


def build_tree(data, depth, max_depth):
    node = Node()
    if gini(data) == 0.0 or depth >= max_depth:
        node.prediction = majority_label(data)
        return node
    feature, value, gain = best_split(data)
    if feature is None or gain <= 0.0:
        node.prediction = majority_label(data)
        return node
    node.feature = feature
    node.value = value
    left, right = split(data, feature, value)
    node.left = build_tree(left, depth + 1, max_depth)
    node.right = build_tree(right, depth + 1, max_depth)
    return node


## 6. `predict` + `print_tree` — 트리 시각화

`predict`는 뿌리에서 시작해 잎까지 질문을 따라 내려갑니다. `print_tree`는 그 구조를 들여쓰기로 보여줍니다.


In [7]:
def predict(node, sample):
    if node.prediction is not None:
        return node.prediction
    if sample["features"][node.feature] == node.value:
        return predict(node.left, sample)
    else:
        return predict(node.right, sample)


def print_tree(node, indent):
    if node.prediction is not None:
        print(indent + "└─ 예측: " + node.prediction)
        return
    print(indent + "[" + node.feature + " == " + node.value + " ?]")
    print(indent + " ├─ YES →")
    print_tree(node.left, indent + " │   ")
    print(indent + " └─ NO  →")
    print_tree(node.right, indent + "     ")


print("=" * 50)
print(" 깊이 2 트리")
print("=" * 50)
tree2 = build_tree(train, 0, 2)
print_tree(tree2, "")


 깊이 2 트리


[odor == n ?]
 ├─ YES →
 │   [spore-print-color == r ?]
 │    ├─ YES →
 │    │   └─ 예측: p
 │    └─ NO  →
 │        └─ 예측: e
 └─ NO  →
     [stalk-root == c ?]
      ├─ YES →
      │   └─ 예측: e
      └─ NO  →
          └─ 예측: p


## 7. `max_depth` 실험 — 깊이별 정확도

깊이를 1, 2, 3, 5로 늘려가며 train/test 정확도를 비교합니다.


In [8]:
def accuracy(node, data):
    correct = 0
    for s in data:
        if predict(node, s) == s["label"]:
            correct = correct + 1
    return correct / len(data)


results = []
for md_depth in [1, 2, 3, 5]:
    tree = build_tree(train, 0, md_depth)
    tr = accuracy(tree, train)
    te = accuracy(tree, test)
    results.append((md_depth, tr, te))

print("+-----------+-------------+------------+")
print("| max_depth | train  acc  | test  acc  |")
print("+-----------+-------------+------------+")
for md_depth, tr, te in results:
    print(f"|    {md_depth:>2d}     |   {tr:.4f}    |  {te:.4f}   |")
print("+-----------+-------------+------------+")


+-----------+-------------+------------+
| max_depth | train  acc  | test  acc  |
+-----------+-------------+------------+
|     1     |   0.8877    |  0.8846   |
|     2     |   0.9524    |  0.9585   |
|     3     |   0.9854    |  0.9848   |
|     5     |   0.9996    |  0.9984   |
+-----------+-------------+------------+


### 시각화 — 순수 파이썬 텍스트 막대그래프

> 외부 라이브러리 없이, 문자(`█`)의 개수만으로 정확도를 막대처럼 표현합니다.
> matplotlib 등 시각화 라이브러리도 쓰지 않습니다 (과제 조건: 외부 라이브러리 0개).

In [9]:
def bar(value, width=40):
    filled = int(round(value * width))
    return "█" * filled + "·" * (width - filled)

print("max_depth\ub294 \uae4a\uc744\uc218\ub85d train\uc774 1.0\uc5d0 \uac00\uae4c\uc6cc\uc9d1\ub2c8\ub2e4 (\uc815\ud655\ub3c4 0.85~1.0 \uad6c\uac04 \ud655\ub300 \ud45c\uc2dc)\n")
lo = 0.85
for md_depth, tr, te in results:
    tr_s = (tr - lo) / (1 - lo)
    te_s = (te - lo) / (1 - lo)
    print(f"depth={md_depth}")
    print(f"  train | {bar(tr_s)} | {tr:.4f}")
    print(f"  test  | {bar(te_s)} | {te:.4f}")
    print()


max_depth는 깊을수록 train이 1.0에 가까워집니다 (정확도 0.85~1.0 구간 확대 표시)

depth=1
  train | ██████████······························ | 0.8877
  test  | █████████······························· | 0.8846

depth=2
  train | ███████████████████████████············· | 0.9524
  test  | █████████████████████████████··········· | 0.9585

depth=3
  train | ████████████████████████████████████···· | 0.9854
  test  | ████████████████████████████████████···· | 0.9848

depth=5
  train | ████████████████████████████████████████ | 0.9996
  test  | ████████████████████████████████████████ | 0.9984



## 8. 한계와 정리

### ✅ 잘 된 점
- 외부 라이브러리 **0개**, 순수 파이썬 문법만으로 의사결정트리 완성
- 알고리즘 6단계(`gini` → `split` → `best_split` → `Node`/`build_tree` → `predict` → `print_tree`) 모두 구현
- Mushroom 데이터에서 깊이 5만으로 **test 99.8%** 달성

### ⚠️ 의사결정트리의 한계 (교과서적)
1. **과적합**: 깊이를 무제한으로 두면 train은 100%지만 test가 떨어지는 게 일반적.
   - 이 데이터는 워낙 깨끗(범주형, 결측 없음, 식용/독성이 특징과 강하게 연관)해서
     과적합이 약하게 나타남 → 깊이 5에서도 test 0.998.
2. **불안정성**: 데이터가 조금만 바뀌어도 트리 구조가 크게 바뀜.
3. **연속형 특징 처리 X**: 우리 구현은 범주형 `==` 비교만 지원.
4. **정보이득의 욕심(greedy)**: 매 분기마다 국소 최적만 봄 → 전역 최적은 보장 X.

### 🛠 발표 디펜스 포인트
- **왜 odor가 첫 분기?** → 단일 특징 중 식용/독성을 가장 깔끔하게 가르는 변수.
  냄새가 없으면(`n`) 거의 식용, 있으면(이 외) 거의 독성.
- **gini 대신 entropy?** → 거의 비슷. gini가 계산이 약간 더 쌈 (제곱만, 로그 X).
- **train/test 분할이 왜 i%10<7?** → 라이브러리 없이도 결정론적·재현 가능하게 7:3 만들기 위함.

---

_끝까지 봐주셔서 감사합니다. — 16조_
